# Notebook 3: Multi-Layer Geospatial Visualization & Mapping

This notebook builds interactive HTML maps using `folium` to visually validate our smartphone IMU-based IRI estimation model against physical road characteristics in the PVS dataset. For each trip, we generate an interactive map featuring three distinct layers:
1. **Predicted IRI Severity Band (Layer 1):** Color-coded circle markers representing our TFLite model's continuous roughness estimates categorized by our optimal grid-searched boundaries ($T_1^*, T_2^*$):
   - **Green (Good):** $\text{IRI} < T_1^*$
   - **Orange/Yellow (Regular):** $T_1^* \le \text{IRI} < T_2^*$
   - **Red (Bad):** $\text{IRI} \ge T_2^*$
2. **Ground Truth PVS Road Quality (Layer 2):** Color-coded track indicating the manual visual/vehicle inspection labels (`Good = 0`, `Regular = 1`, `Bad = 2`).
3. **Road Surface Textures & Physical Anomalies (Layer 3):** Distinctive markers and popups highlighting transitions across pavement types (`asphalt`, `cobblestone`, `dirt`) and physical obstacles (`speed bumps`).

These self-contained HTML maps allow interactive layer toggling, zooming, and tooltip inspection, providing compelling qualitative proof for journal reviewers of how our physics-invariant model responds to real-world road anomalies.


In [1]:
import os
import json
import pandas as pd
import numpy as np
import folium

# --- CONFIGURATION & PATHS ---
DATA_DIR = r"D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\data"
OUTPUT_DIR = r"D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs"
MAPS_DIR = os.path.join(OUTPUT_DIR, "maps")
os.makedirs(MAPS_DIR, exist_ok=True)

# Load optimal thresholds from Notebook 2 (fallback to 3.5 and 7.0 if not found)
thresh_file = os.path.join(OUTPUT_DIR, "optimal_thresholds.json")
if os.path.exists(thresh_file):
    with open(thresh_file, 'r') as f:
        thresh_data = json.load(f)
    T1 = thresh_data.get('T1', 3.5)
    T2 = thresh_data.get('T2', 7.0)
    print(f"[*] Loaded Optimal Thresholds -> T1: {T1} m/km | T2: {T2} m/km")
else:
    T1, T2 = 3.5, 7.0
    print(f"[!] optimal_thresholds.json not found. Using default thresholds -> T1: {T1} m/km | T2: {T2} m/km")


[*] Loaded Optimal Thresholds -> T1: 3.8 m/km | T2: 11.7 m/km


In [2]:
# --- COLOR & ANOMALY HELPERS ---
def get_iri_color(iri_val):
    if iri_val < T1:
        return 'green', 'Good Road'
    elif iri_val < T2:
        return 'orange', 'Regular Road'
    else:
        return 'red', 'Bad Road'

def get_gt_color(label):
    if label == 0:
        return '#2ca02c', 'Good (0)'
    elif label == 1:
        return '#ff7f0e', 'Regular (1)'
    else:
        return '#d62728', 'Bad (2)'


In [3]:
# --- MAP GENERATOR FUNCTION ---
def generate_trip_map(trip_id):
    win_csv = os.path.join(OUTPUT_DIR, "all_trips_windowed_evaluation.csv")
    raw_csv = os.path.join(DATA_DIR, f"PVS_{trip_id}_with_predictions.csv")
    
    if not (os.path.exists(win_csv) and os.path.exists(raw_csv)):
        print(f"[!] Skipping PVS {trip_id}: Missing input files.")
        return None
        
    df_all_win = pd.read_csv(win_csv)
    df_win = df_all_win[df_all_win['trip_id'] == f"PVS {trip_id}"].copy()
    df_raw = pd.read_csv(raw_csv)
    
    if len(df_win) == 0 or len(df_raw) == 0:
        print(f"[!] No valid window data for PVS {trip_id}.")
        return None
        
    print(f"[+] Generating multi-layer map for PVS {trip_id} ({len(df_win):,} windows)...")
    
    # Initialize map centered at route start
    start_lat = df_win['mean_lat'].iloc[0]
    start_lon = df_win['mean_lon'].iloc[0]
    m = folium.Map(location=[start_lat, start_lon], zoom_start=15, tiles='CartoDB positron')
    
    # Feature Groups for Layer Control
    fg_iri = folium.FeatureGroup(name=f"Layer 1: Predicted IRI Bins (T1={T1}, T2={T2})", show=True)
    fg_gt = folium.FeatureGroup(name="Layer 2: Ground Truth Road Quality", show=True)
    fg_surface = folium.FeatureGroup(name="Layer 3: Surface Types & Speed Bumps", show=True)
    
    # 1. Add Predicted IRI Layer
    for idx, row in df_win.iterrows():
        color, class_name = get_iri_color(row['predicted_iri'])
        tooltip_text = (
            f"<b>Predicted IRI:</b> {row['predicted_iri']:.2f} m/km<br>"
            f"<b>Predicted Severity:</b> {class_name}<br>"
            f"<b>Mean Speed:</b> {row['mean_speed_kmh']:.1f} km/h<br>"
            f"<b>Ground Truth Class:</b> {row['label_mid']}"
        )
        folium.CircleMarker(
            location=[row['mean_lat'], row['mean_lon']],
            radius=5,
            weight=1,
            color='black',
            fill=True,
            fill_color=color,
            fill_opacity=0.85,
            tooltip=tooltip_text
        ).add_to(fg_iri)
        
    # 2. Add Ground Truth Layer (Smaller radius for visual overlay comparison)
    for idx, row in df_win.iterrows():
        gt_color, gt_name = get_gt_color(row['label_mid'])
        folium.CircleMarker(
            location=[row['mean_lat'], row['mean_lon']],
            radius=2.5,
            weight=0,
            fill=True,
            fill_color=gt_color,
            fill_opacity=0.9,
            tooltip=f"Ground Truth: {gt_name}"
        ).add_to(fg_gt)
        
    # 3. Add Surface Anomalies Layer (Scan raw df for transitions and obstacles)
    df_sub = df_raw.iloc[::50].copy()
    
    for idx, row in df_sub.iterrows():
        lat, lon = row['latitude'], row['longitude']
        if pd.isna(lat) or pd.isna(lon):
            continue
            
        # Check Speed Bumps
        if row.get('speed_bump_asphalt', 0) == 1 or row.get('speed_bump_cobblestone', 0) == 1:
            bump_type = "Cobblestone Speed Bump" if row.get('speed_bump_cobblestone', 0) == 1 else "Asphalt Speed Bump"
            iri_str = f"{row['predicted_iri']:.2f}" if pd.notna(row.get('predicted_iri', np.nan)) else "N/A"
            folium.Marker(
                location=[lat, lon],
                icon=folium.Icon(color='red', icon='exclamation-triangle', prefix='fa'),
                popup=f"<b>Obstacle:</b> {bump_type}<br><b>Local Predicted IRI:</b> {iri_str} m/km",
                tooltip=f"⚠️ {bump_type}"
            ).add_to(fg_surface)
            
        # Check Cobblestone sections
        elif row.get('cobblestone_road', 0) == 1 and idx % 250 == 0:
            folium.CircleMarker(
                location=[lat, lon],
                radius=8,
                color='purple',
                fill=True,
                fill_color='purple',
                fill_opacity=0.4,
                tooltip="🧱 Cobblestone Pavement Zone"
            ).add_to(fg_surface)
            
        # Check Dirt / Unpaved sections
        elif (row.get('dirt_road', 0) == 1 or row.get('unpaved_road', 0) == 1) and idx % 250 == 0:
            folium.CircleMarker(
                location=[lat, lon],
                radius=8,
                color='brown',
                fill=True,
                fill_color='brown',
                fill_opacity=0.5,
                tooltip="🟤 Dirt / Unpaved Road Zone"
            ).add_to(fg_surface)
            
    m.add_child(fg_iri)
    m.add_child(fg_gt)
    m.add_child(fg_surface)
    folium.LayerControl(collapsed=False).add_to(m)
    
    out_map_path = os.path.join(MAPS_DIR, f"PVS_{trip_id}_interactive_map.html")
    m.save(out_map_path)
    print(f"    [✔] Exported interactive map -> {out_map_path}")
    return out_map_path


In [4]:
# --- GENERATE MAPS FOR ALL 9 TRIPS ---
print("="*80)
print("EXPORTING MULTI-LAYER INTERACTIVE HTML MAPS")
print("="*80)

generated_maps = []
for i in range(1, 10):
    map_path = generate_trip_map(i)
    if map_path:
        generated_maps.append(map_path)

print(f"\n[✔] Successfully generated {len(generated_maps)} trip maps in {MAPS_DIR}")


EXPORTING MULTI-LAYER INTERACTIVE HTML MAPS
[+] Generating multi-layer map for PVS 1 (1,368 windows)...
    [✔] Exported interactive map -> D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\PVS_1_interactive_map.html
[+] Generating multi-layer map for PVS 2 (1,144 windows)...
    [✔] Exported interactive map -> D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\PVS_2_interactive_map.html
[+] Generating multi-layer map for PVS 3 (1,060 windows)...
    [✔] Exported interactive map -> D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\PVS_3_interactive_map.html
[+] Generating multi-layer map for PVS 4 (1,368 windows)...
    [✔] Exported interactive map -> D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\PVS_4_interactive_map.html
[+] Generating multi-layer map for PVS 5 (1,145 windows)...
    [✔] Exported interactive map -> D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\PVS_5_interactive_map.html
[+] Generat

In [5]:
# --- GENERATE COMBINED OVERVIEW MAP ---
print("[+] Constructing global PVS overview map...")
win_csv = os.path.join(OUTPUT_DIR, "all_trips_windowed_evaluation.csv")
if os.path.exists(win_csv):
    df_all_win = pd.read_csv(win_csv)
    
    mean_lat = df_all_win['mean_lat'].mean()
    mean_lon = df_all_win['mean_lon'].mean()
    m_global = folium.Map(location=[mean_lat, mean_lon], zoom_start=13, tiles='CartoDB positron')
    
    df_global_sub = df_all_win.iloc[::2].copy()
    
    fg_global_iri = folium.FeatureGroup(name="Global Predicted IRI Severity", show=True)
    for idx, row in df_global_sub.iterrows():
        color, class_name = get_iri_color(row['predicted_iri'])
        folium.CircleMarker(
            location=[row['mean_lat'], row['mean_lon']],
            radius=4,
            weight=0,
            fill=True,
            fill_color=color,
            fill_opacity=0.75,
            tooltip=f"{row['trip_id']} | IRI: {row['predicted_iri']:.2f} m/km ({class_name})"
        ).add_to(fg_global_iri)
        
    m_global.add_child(fg_global_iri)
    folium.LayerControl().add_to(m_global)
    
    global_map_path = os.path.join(MAPS_DIR, "combined_pvs_overview_map.html")
    m_global.save(global_map_path)
    print(f"[✔] Global combined overview map exported to: {global_map_path}")


[+] Constructing global PVS overview map...
[✔] Global combined overview map exported to: D:\Coding\Hackathon\GFG\ARM\ARM\ml_model\review_work\outputs\maps\combined_pvs_overview_map.html
